# BOOK SCRAPPER

## STEP 1: Importing Libraries and Connect to the Website

In [4]:
# import the libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
# specify the URL of the website to scrape
link = 'https://books.toscrape.com/catalogue/page-1.html'
# send a GET request to the website
response = requests.get(link)
# crete a BeautifulSoup object to parse the HTML content
soup = BeautifulSoup(response.text, 'html.parser')

## STEP 2: Scrapping The Frist Page

In [5]:
book = soup.find_all('li',class_ = 'col-xs-6 col-sm-4 col-md-3 col-lg-3')
data=[]

for sp in soup.find_all('li',class_ = 'col-xs-6 col-sm-4 col-md-3 col-lg-3'):
	#Finding different aspects to be scraped from the first page 
    book_link   = "https://books.toscrape.com/catalogue/" + sp.find_all('a')[-1].get('href')
    
    title       = sp.find_all('a')[-1].get('title')
    
    img_link    = "https://books.toscrape.com/" + sp.find('img').get('src')[3:]
    
    book_rating = (sp.find('p').get('class')[-1])
    
    price       = sp.find('p',class_='price_color').text[1:]
    
    stock       =  sp.find('p',class_ = 'instock availability').text.strip()
	#Appending all the data into a list(data)
    data.append([title,book_rating,price,stock,book_link,img_link])
#Creating a dataframe using pandas library
df = pd.DataFrame(data,columns=['Title','Rating','Price','Stock','Book Link','Image Link'])
df.head()

,Title,Rating,Price,Stock,Book Link,Image Link
0,A Light in the Attic,Three,£51.77,In stock,https://books.toscrape.com/catalogue/a-light-i...,https://books.toscrape.com/media/cache/2c/da/2...
1,Tipping the Velvet,One,£53.74,In stock,https://books.toscrape.com/catalogue/tipping-t...,https://books.toscrape.com/media/cache/26/0c/2...
2,Soumission,One,£50.10,In stock,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/media/cache/3e/ef/3...
3,Sharp Objects,Four,£47.82,In stock,https://books.toscrape.com/catalogue/sharp-obj...,https://books.toscrape.com/media/cache/32/51/3...
4,Sapiens: A Brief History of Humankind,Five,£54.23,In stock,https://books.toscrape.com/catalogue/sapiens-a...,https://books.toscrape.com/media/cache/be/a5/b...


## STEP 3:Scrappe Multplie Pages

In [6]:
from tqdm import tqdm
Multiple_Pages = []
#tqdm used for better representation.
for page in tqdm(range(1,51)):
  	#using a for loop as there are 50 pages then creating a link using page-1,page-2...page-50
    link = 'https://books.toscrape.com/catalogue/page-'+str(page)+'.html'
    res = requests.get(link)
    soap = BeautifulSoup(res.text,'html.parser')
    
    #same code as in scraping for 1 page
    for sp in soap.find_all('li',class_ = 'col-xs-6 col-sm-4 col-md-3 col-lg-3'):
        book_link   = "https://books.toscrape.com/catalogue/" + sp.find_all('a')[-1].get('href')
        title       = sp.find_all('a')[-1].get('title')
        img_link    = "https://books.toscrape.com/" + sp.find('img').get('src')[3:]
        book_rating = (sp.find('p').get('class')[-1])
        price       = sp.find('p',class_='price_color').text[1:]
        stock       =  sp.find('p',class_ = 'instock availability').text.strip()
        Multiple_Pages.append([title,book_rating,price,stock,book_link,img_link])


#Creating a Dataframe
Multiple_Pages_df = pd.DataFrame(data=Multiple_Pages)
Multiple_Pages_df = Multiple_Pages_df.rename(columns={0: 'Title', 1: 'Rating',2:'Price',3:'Stock Available',4:'Book Link',5:'Image Link'})
df_1 = Multiple_Pages_df

#Exporting the Dataframe to an Excel file
Multiple_Pages_df.to_excel('All Books.xlsx', index=False)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [01:26<00:00,  1.72s/it]


## STEP 4:Scrappe Book Deatils

In [9]:
from tqdm import tqdm
# code
data_2 = []
for links in tqdm(Multiple_Pages_df['Book Link']):
    res        = requests.get(links)
    soup       = BeautifulSoup(res.text,'html.parser')  
    Book_Genre       = soup.find('ul',class_ = 'breadcrumb').find_all('a')[2].text
    temp       = soup.find('table',class_ = 'table table-striped').find_all('td')    
    UPC              = temp[0].text
    price_exc_tax    = temp[2].text[1:]
    price_inc_tax    = temp[3].text[2:]
    tax              = temp[4].text[2:]
    availability     = temp[5].text
    reviews          = temp[6].text   
    data_2.append([Book_Genre,price_exc_tax,price_inc_tax,tax,UPC,availability,reviews])

#Creating a dataframe
df_2 = pd.DataFrame(data_2)
df_2 = df_2.rename(columns={0:'Genre',1:'Price',2:"Price(Excluding Tax)",3:"Price(Including Tax)" ,4:"UPC",5:"Stock",6:"Reviews"})

#Saving Dataframe
df_2.to_excel('All Page.xlsx',index=False)

df = pd.DataFrame()
df['UPC'] = df_2['UPC']
df['Title'] = df_1['Title']
df['Genre'] = df_2['Genre']
df['Rating'] = df_1['Rating']
df['Price'] = df_2['Price']
df['Total Tax'] = df_2["Price(Including Tax)"]
df['Stock'] = df_2['Stock']
df['Review'] = df_2['Reviews']
df['Book Link'] = df_1['Book Link']
df['Cover Image'] = df_1['Image Link']
df.to_excel('Final_Books_Data.xlsx',index=False)

df.head()

100%|██████████| 1000/1000 [22:49<00:00,  1.37s/it]


,UPC,Title,Genre,Rating,Price,Total Tax,Stock,Review,Book Link,Cover Image
0,a897fe39b1053632,A Light in the Attic,Poetry,Three,£51.77,0.00,In stock (22 available),0,https://books.toscrape.com/catalogue/a-light-i...,https://books.toscrape.com/media/cache/2c/da/2...
1,90fa61229261140a,Tipping the Velvet,Historical Fiction,One,£53.74,0.00,In stock (20 available),0,https://books.toscrape.com/catalogue/tipping-t...,https://books.toscrape.com/media/cache/26/0c/2...
2,6957f44c3847a760,Soumission,Fiction,One,£50.10,0.00,In stock (20 available),0,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/media/cache/3e/ef/3...
3,e00eb4fd7b871a48,Sharp Objects,Mystery,Four,£47.82,0.00,In stock (20 available),0,https://books.toscrape.com/catalogue/sharp-obj...,https://books.toscrape.com/media/cache/32/51/3...
4,4165285e1663650f,Sapiens: A Brief History of Humankind,History,Five,£54.23,0.00,In stock (20 available),0,https://books.toscrape.com/catalogue/sapiens-a...,https://books.toscrape.com/media/cache/be/a5/b...
